# Notebook 02 — XLM-R Focal Loss (γ=1.0) Multi-Seed Variance Analysis

**Purpose:** Establish a 95% confidence interval for the best model (XLM-R Focal γ=1.0)  
on the 1,700-sample clean test set. Addresses reviewer request for statistical significance  
of the transformer–SVM gap.

**Seeds:** 42, 123, 456 (same as XLM-R standard multi-seed run for direct comparison).  
**Expected runtime:** ~3–4 hours on RTX 4060 Ti (3 seeds × 7 epochs each).

In [1]:
import sys
import sys
from pathlib import Path
_cwd = Path.cwd()
for _p in [_cwd / '../src', _cwd / '../3_Source', _cwd / 'src', _cwd / '3_Source']:
    if (_p / 'config.py').exists():
        sys.path.insert(0, str(_p.resolve()))
        break
else:
    raise ImportError('config.py not found -- run Jupyter from the project root or notebooks/ directory')

import json, random, os
import numpy as np
import torch
from pathlib import Path
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, DataCollatorWithPadding,
    set_seed,
)
from datasets import Dataset
import pandas as pd
from sklearn.metrics import classification_report, f1_score

from config import (
    RESULTS_DIR, CLEAN_TEST_CSV, CLEAN_DEV_CSV,
    LABEL2ID, ID2LABEL,
)

CHECKPOINT          = 'xlm-roberta-base'
GAMMA               = 1.0
SEEDS               = [42, 123, 456]
FOCAL_MULTISEED_DIR = RESULTS_DIR / 'focal_multiseed'
FOCAL_MULTISEED_DIR.mkdir(exist_ok=True)

print(f'Seeds: {SEEDS}  |  Focal γ={GAMMA}  |  Checkpoint: {CHECKPOINT}')
print(f'Output dir: {FOCAL_MULTISEED_DIR}')

Seeds: [42, 123, 456]  |  Focal γ=1.0  |  Checkpoint: xlm-roberta-base
Output dir: D:\Doc\Programming in AI\Project\results\focal_multiseed


## Focal Loss Classes

Copied verbatim from `02_focal_loss.ipynb`. Do not modify.

In [2]:
import torch.nn as nn
import torch.nn.functional as F
from transformers import Trainer

class FocalLoss(nn.Module):
    """
    Focal Loss for multi-class classification.

    Addresses class imbalance by down-weighting easy (well-classified) examples
    and focusing gradient on hard misclassified examples. Critical for the
    MESocSentiment corpus where Positive class is only 11.3% of test set.

    Reference: Lin et al. (2017) "Focal Loss for Dense Object Detection"
    gamma=0: equivalent to standard cross-entropy
    gamma=1: moderate focusing
    gamma=2: standard setting from original paper, strong focusing on hard examples
    """
    def __init__(self, gamma: float = 2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        # logits: (batch_size, num_classes)
        # labels: (batch_size,) — class indices, not one-hot
        ce_loss   = F.cross_entropy(logits, labels, reduction='none')   # (batch,)
        pt        = torch.exp(-ce_loss)                                  # p_t: probability of true class
        focal_loss = (1.0 - pt) ** self.gamma * ce_loss                  # focal modulation
        return focal_loss.mean()


class FocalTrainer(Trainer):
    """
    HuggingFace Trainer with Focal Loss replacing standard cross-entropy.
    """
    def __init__(self, focal_gamma: float = 2.0, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_loss_fn = FocalLoss(gamma=focal_gamma)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        loss    = self.focal_loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


print(f'FocalLoss and FocalTrainer defined.')
print(f'  gamma=0 \u2192 standard cross-entropy')
print(f'  gamma=1 \u2192 moderate focusing on hard examples')
print(f'  gamma=2 \u2192 strong focusing (standard from Lin et al. 2017)')


from torch.optim import AdamW


class FocalDifferentialLRTrainer(FocalTrainer):
    """FocalTrainer + differential learning rates (head=2e-5, backbone=5e-6)."""

    def create_optimizer(self):
        no_decay      = {"bias", "LayerNorm.weight", "layer_norm.weight", "norm.weight"}
        head_keywords = {"classifier", "pooler"}

        head_decay, head_no_decay       = [], []
        backbone_decay, backbone_no_decay = [], []

        for name, param in self.model.named_parameters():
            if not param.requires_grad:
                continue
            is_head     = any(kw in name for kw in head_keywords)
            is_no_decay = any(nd in name for nd in no_decay)

            if is_head:
                (head_no_decay if is_no_decay else head_decay).append(param)
            else:
                (backbone_no_decay if is_no_decay else backbone_decay).append(param)

        backbone_lr  = self.args.learning_rate * 0.25
        param_groups = [
            {"params": head_decay,        "lr": self.args.learning_rate, "weight_decay": self.args.weight_decay},
            {"params": head_no_decay,     "lr": self.args.learning_rate, "weight_decay": 0.0},
            {"params": backbone_decay,    "lr": backbone_lr,             "weight_decay": self.args.weight_decay},
            {"params": backbone_no_decay, "lr": backbone_lr,             "weight_decay": 0.0},
        ]
        param_groups = [g for g in param_groups if len(g["params"]) > 0]
        self.optimizer = AdamW(param_groups, eps=1e-8)
        return self.optimizer


print('FocalDifferentialLRTrainer defined.')


FocalLoss and FocalTrainer defined.
  gamma=0 → standard cross-entropy
  gamma=1 → moderate focusing on hard examples
  gamma=2 → strong focusing (standard from Lin et al. 2017)
FocalDifferentialLRTrainer defined.


## Dataset Helpers

In [3]:
import re

def preprocess(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def load_splits():
    # Use train_split.csv — the same 90% stratified split used by 02_focal_loss.ipynb
    # and 02_transformer_v2.ipynb (n=8,915 after deduplication).
    # IMPORTANT: do NOT use TRAIN_CSV directly. TRAIN_CSV has 10,863 rows (with
    # duplicates) and 9,906 after dedup — neither matches the training set used for
    # the single-run peak (0.6841). Using train_split.csv ensures the multi-seed CI
    # characterises the same model configuration as the headline result.
    train_df = pd.read_csv(RESULTS_DIR / 'train_split.csv')
    dev_df   = pd.read_csv(CLEAN_DEV_CSV)
    test_df  = pd.read_csv(CLEAN_TEST_CSV)

    # train_split.csv already has 'text' and 'label' columns (saved by notebook 01)
    if 'text_clean' not in train_df.columns:
        train_df['text_clean'] = train_df['text'].apply(preprocess)

    for df in [dev_df, test_df]:
        if 'text_clean' not in df.columns and 'text' in df.columns:
            df['text_clean'] = df['text'].apply(preprocess)

    print(f'Train: {len(train_df)} Dev: {len(dev_df)} Test: {len(test_df)}')
    print(f'  NOTE: train = train_split.csv (matches focal_loss.ipynb, n=8,915)')
    return train_df, dev_df, test_df


def df_to_hf_dataset(df, tokenizer, max_length=128):
    enc = tokenizer(
        df['text_clean'].astype(str).tolist(),
        truncation=True,
        padding=False,
        max_length=max_length,
    )
    enc['labels'] = [LABEL2ID[l] for l in df['label']]
    return Dataset.from_dict(enc)


def compute_metrics_fn(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'macro_f1': f1_score(labels, preds, average='macro')}

## Multi-Seed Training Loop

In [4]:
TRAINING_ARGS = dict(
    num_train_epochs            = 7,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 64,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'macro_f1',
    fp16                        = True,
    logging_steps               = 50,
    dataloader_num_workers      = 0,
    report_to                   = 'none',
)

seed_results  = []
train_df, dev_df, test_df = load_splits()

for seed in SEEDS:
    print(f'\n{"="*60}')
    print(f'  SEED {seed}')
    print(f'{"="*60}')

    set_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    seed_dir = FOCAL_MULTISEED_DIR / f'seed_{seed}'
    seed_dir.mkdir(exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
    train_ds  = df_to_hf_dataset(train_df, tokenizer)
    dev_ds    = df_to_hf_dataset(dev_df,   tokenizer)
    test_ds   = df_to_hf_dataset(test_df,  tokenizer)
    collator  = DataCollatorWithPadding(tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        CHECKPOINT, num_labels=3,
        id2label=ID2LABEL, label2id=LABEL2ID,
    )

    args = TrainingArguments(
        output_dir = str(seed_dir / 'checkpoints'),
        seed       = seed,
        **TRAINING_ARGS,
    )

    trainer = FocalDifferentialLRTrainer(
        focal_gamma      = GAMMA,
        model            = model,
        args             = args,
        train_dataset    = train_ds,
        eval_dataset     = dev_ds,
        processing_class = tokenizer,
        data_collator    = collator,
        compute_metrics  = compute_metrics_fn,
    )
    trainer.train()

    preds_out = trainer.predict(test_ds)
    preds     = np.argmax(preds_out.predictions, axis=-1)
    labels    = test_df['label'].map(LABEL2ID).values
    report    = classification_report(
        labels, preds,
        target_names=['NEGATIVE', 'NEUTRAL', 'POSITIVE'],
        output_dict=True,
    )

    macro_f1 = report['macro avg']['f1-score']
    result = {
        'seed':     seed,
        'macro_f1': round(macro_f1, 4),
        'per_class': {
            cls: {k: round(v, 4) for k, v in report[cls].items()
                  if k in ('precision', 'recall', 'f1-score')}
            for cls in ('NEGATIVE', 'NEUTRAL', 'POSITIVE')
        },
    }
    seed_results.append(result)
    print(f'\nSeed {seed}  →  Macro-F1 = {macro_f1:.4f}')

    with open(seed_dir / 'results.json', 'w') as f:
        json.dump(result, f, indent=2)

print('\n=== All seeds complete ===')
for r in seed_results:
    print(f'  Seed {r["seed"]}: {r["macro_f1"]}')

Train: 8915 Dev: 300 Test: 1700
  NOTE: train = train_split.csv (matches focal_loss.ipynb, n=8,915)

  SEED 42


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Macro F1
1,0.246871,0.836084,0.617885
2,0.157607,0.453713,0.722928
3,0.116630,0.660129,0.702634
4,0.089836,0.769992,0.714910
5,0.040718,0.865834,0.719556
6,0.050179,1.035326,0.692939
7,0.030991,1.053551,0.700800


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Seed 456  →  Macro-F1 = 0.6787

=== All seeds complete ===
  Seed 42: 0.6734
  Seed 123: 0.6724
  Seed 456: 0.6787


In [5]:
per_seed_f1 = [r['macro_f1'] for r in seed_results]
mean_f1     = float(np.mean(per_seed_f1))
std_f1      = float(np.std(per_seed_f1, ddof=1))

# 95% CI using t-distribution (n=3, df=2, t_crit=4.303)
# Correct formula: mean ± t * SE  where SE = std / sqrt(n)
N_SEEDS    = len(per_seed_f1)          # 3
T_CRIT     = 4.303                     # t(0.025, df=2)
se_f1      = std_f1 / np.sqrt(N_SEEDS)
ci95_lower = round(mean_f1 - T_CRIT * se_f1, 4)
ci95_upper = round(mean_f1 + T_CRIT * se_f1, 4)

summary = {
    'model':            'XLM-R (Focal γ=1.0)',
    'gamma':            GAMMA,
    'seeds':            SEEDS,
    'per_seed_results': seed_results,
    'mean_macro_f1':    round(mean_f1, 4),
    'std_macro_f1':     round(std_f1,  4),
    'ci_95_lower':      ci95_lower,
    'ci_95_upper':      ci95_upper,
    'ci_note':          f'95% CI via t-distribution (df={N_SEEDS-1}, t={T_CRIT}); SE=std/sqrt({N_SEEDS})',
}

out_path = FOCAL_MULTISEED_DIR / 'multiseed_results.json'
with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f'Seeds:      {SEEDS}')
print(f'Per-seed:   {per_seed_f1}')
print(f'Mean ± Std: {mean_f1:.4f} ± {std_f1:.4f}')
print(f'SE:         {se_f1:.4f}  (std / sqrt({N_SEEDS}))')
print(f'95% CI:     [{ci95_lower:.4f}, {ci95_upper:.4f}]  (t={T_CRIT}, df={N_SEEDS-1})')
print(f'SVM gap:    {mean_f1 - 0.6407:+.4f}  (SVM baseline = 0.6407)')
print(f'\nSaved → {out_path}')


Seeds:      [42, 123, 456]
Per-seed:   [0.6734, 0.6724, 0.6787]
Mean ± Std: 0.6748 ± 0.0034
SE:         0.0020  (std / sqrt(3))
95% CI:     [0.6664, 0.6832]  (t=4.303, df=2)
SVM gap:    +0.0341  (SVM baseline = 0.6407)

Saved → D:\Doc\Programming in AI\Project\results\focal_multiseed\multiseed_results.json
